In [ ]:
"""
Hyperparameter tuning is the process of selecting the optimal values of training parameters before a neural network begins learning.
Hyperparameters such as the learning rate, batch size, number of epochs, dropout rate and regularization strength are configured prior to training and
have a significant impact on the model's ability to generalize.
"""

In [3]:
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

In [4]:
# Load the Dataset

# if you have not downloaded dataset uncomment and use this
# (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# x_train = x_train.astype("float32") / 255.0
# x_test = x_test.astype("float32") / 255.0           


import numpy as np
import struct
import os

def load_mnist_images(filename):
    with open(filename, 'rb') as f:
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.reshape(num, rows, cols)

def load_mnist_labels(filename):
    with open(filename, 'rb') as f:
        magic, num = struct.unpack('>II', f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data

data_dir = os.path.join('data', 'MNIST', 'raw')

x_train = load_mnist_images(os.path.join(data_dir, 'train-images-idx3-ubyte'))
y_train = load_mnist_labels(os.path.join(data_dir, 'train-labels-idx1-ubyte'))
x_test  = load_mnist_images(os.path.join(data_dir, 't10k-images-idx3-ubyte'))
y_test  = load_mnist_labels(os.path.join(data_dir, 't10k-labels-idx1-ubyte'))

# Normalize
x_train = x_train / 255.0
x_test  = x_test / 255.0

In [5]:
# Create the Neural Network

def create_model(learning_rate=0.001, dropout_rate=0.5):

    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(128, activation="relu"),
        Dropout(dropout_rate),
        Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [6]:
# Define the Hyperparameter Grid

model = KerasClassifier(
    model=create_model,
    verbose=0
)

param_grid = {
    "model__learning_rate": [0.001, 0.0001],
    "model__dropout_rate": [0.3, 0.5],
    "epochs": [5, 10],
    "batch_size": [32, 64]
}


In [8]:
# Perform Grid Search

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3
)

grid_result = grid.fit(x_train, y_train)

/home/soheil/DeepLearning /myvenv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/soheil/DeepLearning /myvenv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/soheil/DeepLearning /myvenv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/soheil/DeepLearning /myvenv/lib/python3.

In [9]:
# Display the Best Hyperparameters

print("Best Parameters:", grid_result.best_params_)
print("Best Score:", grid_result.best_score_)

Best Parameters: {'batch_size': 64, 'epochs': 10, 'model__dropout_rate': 0.3, 'model__learning_rate': 0.001}
Best Score: 0.9734666666666666


In [10]:
# Evaluate the Best Model

best_model = grid_result.best_estimator_.model_

test_loss, test_accuracy = best_model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", test_accuracy)


Test Accuracy: 0.9790999889373779
